# ReAct Agent in LangGraph

**ReAct** = **Reason** + **Act**. An agent alternates between thinking (reasoning) and calling tools (acting) until it can answer.

See [README_REACT.md](../README_REACT.md) for a full explanation. This notebook has three examples:
- **Example 1:** Math + Research (add, multiply, paper lookup)
- **Example 2:** Weather + Calculator (simpler, different tools)
- **Example 3:** Minimal Math (find_sum, find_product) — system prompt forces tool use

## Setup

Ensure you have `langgraph`, `langchain`, `langchain-openai`, and `python-dotenv` installed. Set `OPENAI_API_KEY` in `.env`.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

---
## Example 1: Math + Research Agent

**Tools:** `add_numbers`, `multiply_numbers`, `get_paper_info`

**ReAct flow:** User asks → Agent reasons → Calls tool(s) → Gets results → Agent reasons again → Final answer

In [ ]:
# Define tools. The LLM reads docstrings to decide when to use each.
@tool
def add_numbers(a: float, b: float) -> float:
    """Add two numbers. Use for any addition."""
    return a + b

@tool
def multiply_numbers(a: float, b: float) -> float:
    """Multiply two numbers. Use for any multiplication."""
    return a * b

@tool
def get_paper_info(paper_title: str) -> str:
    """Look up a research paper by title. Use when asked about academic papers."""
    mock = {
        "attention is all you need": "Vaswani et al. (2017): Transformer architecture, attention-only.",
        "bert": "Devlin et al. (2018): BERT, bidirectional pre-training for NLP.",
    }
    key = paper_title.lower().strip()
    return mock.get(key, f"Paper '{paper_title}' not found.")

In [ ]:
# Build the ReAct graph: agent <-> tools loop
tools = [add_numbers, multiply_numbers, get_paper_info]
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm_with_tools = llm.bind_tools(tools)

def agent_node(state: MessagesState):
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode(tools))
graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", tools_condition)
graph.add_edge("tools", "agent")

app1 = graph.compile()

In [ ]:
# Example 1a: Math
result = app1.invoke({"messages": [HumanMessage(content="What is 127 × 34?")]})
print("Answer:", result["messages"][-1].content)

In [ ]:
# Example 1b: Research paper
result = app1.invoke({"messages": [HumanMessage(content="Tell me about the paper 'Attention is All You Need'")]})
print("Answer:", result["messages"][-1].content)

In [ ]:
# Example 1c: Multi-step (math + paper in one query)
result = app1.invoke({"messages": [HumanMessage(content="What is 15 + 27? And summarize BERT.")]})
print("Answer:", result["messages"][-1].content)

---
## Example 2: Weather + Calculator Agent

**Tools:** `get_weather`, `add` — a simpler agent for weather and arithmetic.

In [ ]:
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city. Use when asked about weather."""
    # Mock — in production, call a real API
    return f"The weather in {city} is sunny, 72°F."

@tool
def add(a: int, b: int) -> int:
    """Add two integers. Use for arithmetic."""
    return a + b

In [ ]:
# Build Example 2 agent
tools2 = [get_weather, add]
llm2 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm2_with_tools = llm2.bind_tools(tools2)

def agent_node2(state: MessagesState):
    response = llm2_with_tools.invoke(state["messages"])
    return {"messages": [response]}

graph2 = StateGraph(MessagesState)
graph2.add_node("agent", agent_node2)
graph2.add_node("tools", ToolNode(tools2))
graph2.add_edge(START, "agent")
graph2.add_conditional_edges("agent", tools_condition)
graph2.add_edge("tools", "agent")

app2 = graph2.compile()

In [ ]:
# Example 2a: Weather only
result = app2.invoke({"messages": [HumanMessage(content="What's the weather in Paris?")]})
print("Answer:", result["messages"][-1].content)

In [ ]:
# Example 2b: Weather + math in one query
result = app2.invoke({"messages": [HumanMessage(content="What's the weather in Tokyo and what is 10 + 20?")]})
print("Answer:", result["messages"][-1].content)

---
## Summary

| Component | Role |
|-----------|------|
| `@tool` | Turns a function into a tool the LLM can call |
| `bind_tools` | Gives the LLM awareness of available tools |
| `ToolNode` | Executes tool_calls and returns results |
| `tools_condition` | Routes: tool_calls → tools, else → END |
| System prompt | Shapes agent behavior (e.g. "use only tools") |

The ReAct loop: **Agent → (tool_calls?) → Tools → Agent → ... → END**

---
## Example 3: Minimal Math Agent (find_sum + find_product)

**Tools:** `find_sum`, `find_product` — system prompt forces the agent to use tools, not compute mentally.

**Key idea:** The prompt says *"Solve by using ONLY the tools available. Do NOT solve the problem yourself"* — so every math question triggers a tool call.

In [ ]:
from langchain_core.messages import SystemMessage

@tool
def find_sum(x: int, y: int) -> int:
    """Add two numbers and return their sum. Use for any addition."""
    return x + y

@tool
def find_product(x: int, y: int) -> int:
    """Multiply two numbers and return their product. Use for any multiplication."""
    return x * y

# System prompt forces tool use — agent must NOT compute in its head
system_prompt = SystemMessage(content="""You are a Math genius who can solve math problems.
Solve the problems by using ONLY the tools available. Do NOT solve the problem yourself.""")

tools3 = [find_sum, find_product]
llm3 = ChatOpenAI(model="gpt-4o-mini", temperature=0)
llm3_with_tools = llm3.bind_tools(tools3)

def agent_node3(state: MessagesState):
    messages = [system_prompt] + state["messages"]
    return {"messages": [llm3_with_tools.invoke(messages)]}

graph3 = StateGraph(MessagesState)
graph3.add_node("agent", agent_node3)
graph3.add_node("tools", ToolNode(tools3))
graph3.add_edge(START, "agent")
graph3.add_conditional_edges("agent", tools_condition)
graph3.add_edge("tools", "agent")

app3 = graph3.compile()

In [ ]:
# Example 3a: Sum
result = app3.invoke({"messages": [HumanMessage(content="what is the sum of 2 and 3?")]})
print("Agent returned:", result["messages"][-1].content)
print("\nStep-by-step execution:")
for msg in result["messages"]:
    print(msg.pretty_repr())

In [ ]:
# Example 3b: Parallel tool calls (product + sum in one turn)
result = app3.invoke({"messages": [HumanMessage(content="What is 3 multiplied by 2 and 5 + 1?")]})
print("Agent returned:", result["messages"][-1].content)
print("\nStep-by-step execution:")
for msg in result["messages"]:
    print(msg.pretty_repr())